# Recurrent amps
Find interesting genomic ranges:
- how many regions amplified in more than *n=3* tumors? (get_recurrent_amps, 109 regions)
- how many with 3 or more oncogenes? (get_oncogene_clusters, 5 clusters)
- what oncogenes are on recurrent amps? (get_recurrently_amp_oncogenes, 41 oncogenes)
- how many without an oncogene? (get_oncogene_deserts, 82 regions)
- how many with genes but no oncogenes? (get_amps_w_genes_no_oncogenes, 38 regions)
- what uninterrupted gene sequences present on the above? (find_whole_genes_in_oncogene_deserts, 81 protein-coding, 235 lncRNA, various others)
- which regions are ecDNA-specific? Intrachromosomal-specific? (get_specific_amps)
- which recurrent amps can be found on whole ecDNA sequences without oncogenes? (13 regions)
- Which genes on the above? (60 protein-coding, 138 lncRNA, various others)

### Requires
- pyranges. See `./pyranges.yml`.
- .bdg outputs from bed-pileup.ipynb.
- .bed output from gene_statistics.ipynb.  

In [ ]:
import pyranges as pr
import pandas as pd
from pathlib import Path
import warnings

import sys
sys.path.append('../../src')
from data_imports import *

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [ ]:
# These operations take awhile so we set global variables
def get_gencode(path='../../data/anno/gencode/gencode.v47.basic.annotation.gff3',include_pseudogenes=False):
    df = pr.read_gff3(path)
    if include_pseudogenes:
        return df[df.Feature == 'gene']
    else:
        return df[(df.Feature == 'gene') & (df.gene_type.isin(['lncRNA','miRNA','misc_RNA','protein_coding','snRNA','snoRNA']))]
def get_oncogene_locations(gencode=None):
    if gencode is None:
        gencode = get_gencode()
    genes = import_genes()
    names = genes[genes.is_canonical_oncogene].gene.values
    oncogene_locations = gencode[gencode.gene_name.isin(names)]
    return oncogene_locations
    
GENCODE47 = get_gencode()
ONCOGENE_LOCATIONS = get_oncogene_locations(GENCODE47)

In [ ]:
GENCODE47.head()

In [ ]:
# Functions to find interesting intervals
def contains(a,b):
    # return all intervals in a which fully contain an interval in b.
    c = b.overlap(a,how='containment')
    return a.overlap(c)
def get_recurrent_amps(path="bedgraph/ecDNA_all.bdg", cn=3, slack=10000):
    # Regions amplified in at least 3 independent tumors. Merge neighboring intervals within 10kb.
    bdg = pr.read_bed(path)
    bdg = bdg[bdg.Name >= cn]
    bdg = bdg.merge(slack=slack)
    return bdg
def get_specific_amps(a,b, cn=3, slack=10000):
    # get recurrent amps in a which are not recurrent amps in b.
    bdga = get_recurrent_amps(path=a,cn=cn,slack=slack)
    bdgb = get_recurrent_amps(path=b,cn=cn,slack=slack)
    bdgb = bdgb.extend(ext=slack)
    return bdga.overlap(bdgb, invert=True)
def get_oncogene_clusters(oncogene_locations=None,cn=3):
    # Recurrent amps with cn or more oncogenes
    if oncogene_locations is None:
        oncogene_locations = get_oncogene_locations()
    recurrent_amps = get_recurrent_amps()
    with warnings.catch_warnings(action="ignore"):
        overlaps = pr.count_overlaps({"count":oncogene_locations},recurrent_amps)
    overlaps = overlaps[overlaps.count >= cn]
    return overlaps
def get_oncogene_deserts(oncogene_locations=None):
    # 'deserts' = recurrent amplifications without an oncogene.
    if oncogene_locations is None:
        oncogene_locations = get_oncogene_locations()
    recurrent_amps = get_recurrent_amps()
    with warnings.catch_warnings(action="ignore"):
        overlaps = pr.count_overlaps({"count":oncogene_locations},recurrent_amps)
    deserts = overlaps[overlaps.count < 1]    
    return deserts
def get_amps_w_genes_no_oncogenes(oncogene_locations=None,gencode=None):
    # recurrent amplifications with genes but no known oncogenes
    if oncogene_locations is None:
        oncogene_locations = get_oncogene_locations()
    if gencode is None:
        gencode = get_gencode()
    deserts = get_oncogene_deserts(oncogene_locations)
    #targets = deserts.overlap(gencode,how='containment')
    targets = contains(deserts,gencode)
    return targets
def find_whole_genes_in_oncogene_deserts(oncogene_locations=None,gencode=None):
    # Genes amplified in the oncogene deserts.
    if oncogene_locations is None:
        oncogene_locations = get_oncogene_locations()
    if gencode is None:
        gencode = get_gencode()
    deserts = get_oncogene_deserts(oncogene_locations)
    targets = gencode.overlap(deserts,how='containment')
    return targets
def get_recurrently_amp_oncogenes(oncogene_locations=None,recurrent_amps=None):
    if oncogene_locations is None:
        oncogene_locations = get_oncogene_locations()
    if recurrent_amps is None:
        recurrent_amps = get_recurrent_amps()
    amp_oncogenes = oncogene_locations.overlap(recurrent_amps,how='containment')
    return amp_oncogenes
def get_ecDNA_sans_oncogenes(path="out/ecDNA_sans_oncogenes.bed"):
    # Subset of ecDNA sequences without a known oncogene.
    bed = pr.read_bed(path)
    return bed
def get_amps_w_genes_no_oncogenic_ecDNA(oncogene_locations=None,gencode=None):
    # recurrent amplifications with genes but no known oncogenes, within ecDNA sequences also without known oncogenes.
    if oncogene_locations is None:
        oncogene_locations = get_oncogene_locations()
    if gencode is None:
        gencode = get_gencode()
    targets = get_amps_w_genes_no_oncogenes(oncogene_locations,gencode)
    #targets = get_oncogene_deserts(oncogene_locations)
    df = targets.overlap(get_ecDNA_sans_oncogenes(),how='containment').merge()
    return df
def find_recurrent_genes_no_oncogenic_ecDNA(oncogene_locations=None,gencode=None):
    # Genes amplified in the oncogene deserts.
    if oncogene_locations is None:
        oncogene_locations = get_oncogene_locations()
    if gencode is None:
        gencode = get_gencode()
    putative_oncogenes = find_whole_genes_in_oncogene_deserts(oncogene_locations,gencode)
    targets = putative_oncogenes.overlap(get_ecDNA_sans_oncogenes(),how='containment')
    return targets

In [ ]:
get_recurrent_amps()

In [ ]:
get_oncogene_clusters(ONCOGENE_LOCATIONS)

In [ ]:
rao_tbl = get_recurrently_amp_oncogenes(ONCOGENE_LOCATIONS)
print(rao_tbl.df.shape)
rao_tbl

In [ ]:
deserts = get_oncogene_deserts(ONCOGENE_LOCATIONS)
deserts.summary()
deserts.head()
deserts.to_bed('tmp/deserts.bed')

In [ ]:
putative_oncoregions = get_amps_w_genes_no_oncogenes(ONCOGENE_LOCATIONS,GENCODE47)
print(putative_oncoregions.summary())
putative_oncoregions.to_bed('tmp/putative_oncoregions.bed')

In [ ]:
independent_oncoregions = get_amps_w_genes_no_oncogenic_ecDNA(ONCOGENE_LOCATIONS,GENCODE47)
print(independent_oncoregions.summary())
independent_oncoregions.to_bed('tmp/independent_oncoregions.bed')

In [ ]:
putative_oncogenes = find_whole_genes_in_oncogene_deserts(ONCOGENE_LOCATIONS,GENCODE47)
putative_oncogenes.df[['gene_name','gene_type']].groupby('gene_type').count()

In [ ]:
independent_oncogenes=find_recurrent_genes_no_oncogenic_ecDNA(ONCOGENE_LOCATIONS,GENCODE47)
#independent_oncogenes
independent_oncogenes.df[['gene_name','gene_type']].drop_duplicates().groupby('gene_type').count()

In [ ]:
putative_oncogenes = putative_oncogenes.df[['Chromosome','Start','End','Strand','gene_id','gene_type',
                                         'gene_name','hgnc_id']]
putative_oncogenes['on_ecDNA_sans_oncogene'] = putative_oncogenes.gene_name.isin(independent_oncogenes.df.gene_name)
putative_oncogenes.to_csv('out/putative_oncogenes.tsv',sep='\t',index=False)
putative_oncogenes[putative_oncogenes.gene_type == 'protein_coding']

In [ ]:
ec_bdg="bedgraph/ecDNA_all.bdg"
ch_bdg="bedgraph/intrachromosomal_all.bdg"
ec_specific_amps = get_specific_amps(ec_bdg,ch_bdg)
get_recurrently_amp_oncogenes(ONCOGENE_LOCATIONS,ec_specific_amps)

In [ ]:
ch_specific_amps = get_specific_amps(ch_bdg,ec_bdg)
get_recurrently_amp_oncogenes(ONCOGENE_LOCATIONS,ch_specific_amps)

In [ ]:
# TEMP
GENCODE47.to_bed('tmp/gencode.bed')